# Data Preparation for Machine Learning

This notebook shows, step by step, how to prepare a dataset **before** building a Machine Learning model.

We use the **Titanic** dataset (available directly in `seaborn`), so there's no need to upload a file: everything works directly in Colab.


In [ ]:
# Importer les librairies nécessaires
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

## Explanation of the imported libraries

- **pandas (pd)**: used to manipulate data in table form (like Excel). It's used to read, filter, and transform the dataset's columns.

- **numpy (np)**: mathematical and scientific computing library. It's often used behind the scenes by other libraries to perform fast numerical calculations.

- **seaborn (sns)**: library used to create charts (data visualization), and it also contains ready-to-use datasets, like the Titanic dataset used here.

- **matplotlib.pyplot (plt)**: the base library for creating charts in Python. Seaborn is built on top of matplotlib.

- **train_test_split** (from sklearn.model_selection): a function that splits the data into two parts: one to train the model, another to test it.

- **StandardScaler** (from sklearn.preprocessing): a tool that puts all numerical variables on the same scale (normalization), so the model doesn't favor a variable just because it has large values (e.g., fare vs age).

- **LabelEncoder** (from sklearn.preprocessing): a tool that converts text categories (e.g., "male"/"female") into numbers (0/1), since a Machine Learning model only understands numbers.


## 1. Load the data


In [ ]:
df = sns.load_dataset('titanic')
df.head()

In [ ]:
df.head()

## Column descriptions

- **survived**: the target to predict → `0` = did not survive, `1` = survived
- **pclass**: ticket class → `1` = 1st class, `2` = 2nd, `3` = 3rd (related to social status)
- **sex**: passenger's sex (`male` or `female`)
- **age**: passenger's age, in years
- **sibsp**: number of siblings/spouses aboard
- **parch**: number of parents/children aboard
- **fare**: price paid for the ticket
- **embarked**: port of embarkation → `S` = Southampton, `C` = Cherbourg, `Q` = Queenstown


## 2. Explore the data

Before touching the data, we need to understand what we have: how many rows/columns, what data types, are there any strange values...


In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
print("Dimensions:", df.shape)
df.info()

### Code explanation
- `df.shape`: returns (number of rows, number of columns) → the size of the dataset at a glance.
- `df.info()`: shows, for each column, its name, the number of non-empty (non-null) values, and its type (`int64`, `float64`, `object`...). Useful for spotting columns with missing values or text.


In [ ]:
df.describe()

### Code explanation
`df.describe()` computes basic statistics for the numerical columns:
- `count`: number of non-empty values
- `mean`: average
- `std`: standard deviation (spread of values)
- `min` / `max`: minimum / maximum value
- `25%`, `50%`, `75%`: quartiles (the `50%` is the median)

This helps spot outliers (e.g., a negative age, an abnormally high price).


## 3. Keep only the useful columns

The Titanic dataset contains redundant columns (e.g., `class` = same info as `pclass`). We keep only what we need.


In [ ]:
df.head(10)

In [ ]:
df = df[['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']]
df.head()

### Code explanation
- `df = df[[...]]`: keeps only the useful columns listed in brackets, and replaces `df` with this reduced version.
- `df.head()`: shows the first 5 rows, to visually check that the selection worked correctly.


## 4. Check for missing values

A Machine Learning model cannot work with empty cells (NaN).


In [ ]:
df.isnull()

In [ ]:
df.isnull().sum()

### Code explanation
- `df.isnull()`: returns a table the same size as `df`, filled with `True`/`False` depending on whether each cell is empty (NaN) or not.
- `.sum()`: adds up the `True` values (counted as 1) column by column → gives the number of missing values per column.


## 5. Handle missing values

- `age` (numerical) → replaced with the **median**
- `embarked` (categorical) → replaced with the most frequent value (**mode**)


In [ ]:
df['age'].median()

In [ ]:
df['embarked'].mode()

In [ ]:
df['age'] = df['age'].fillna(df['age'].median())
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])

In [ ]:

df.isnull().sum()

### Code explanation
- `df['age'].median()`: computes the median value of the `age` column (the middle value once the ages are sorted).
- `.fillna(...)`: replaces all missing values (`NaN`) in the column with the given value.
- Same logic for `embarked`, but with `.mode()[0]`: the mode is the most frequent value; `[0]` takes the first one in case of a tie.
- The last line displays `df.isnull().sum()` again to check that there are no more missing values.


## 6. Encode categorical variables

A Machine Learning model needs **numbers**, not text. So we transform the text columns into numbers.


In [ ]:
data= ["machine","deep","machine","IA"]

In [ ]:
encoder = LabelEncoder()
data=encoder.fit_transform(data)

In [ ]:
data

In [ ]:
le_sex = LabelEncoder()

In [ ]:
df['sex']

In [ ]:
le_sex = LabelEncoder()
df['sex'] = le_sex.fit_transform(df['sex'])

In [ ]:
df['sex']

In [ ]:
# sex : male/female -> 0/1
le_sex = LabelEncoder()
df['sex'] = le_sex.fit_transform(df['sex'])

# embarked : several categories -> one-hot encoding
df = pd.get_dummies(df, columns=['embarked'], drop_first=True)

df.head()

### Code explanation
- `LabelEncoder()` creates a tool that converts text into numbers.
- `.fit_transform(df['sex'])`: learns the categories present (`male`, `female`) and directly replaces them with numbers (0 and 1).
- `pd.get_dummies(df, columns=['embarked'], drop_first=True)`: transforms `embarked` into several binary columns (one per category), with `0` or `1`. `drop_first=True` removes one redundant column (if we know the other two, we can deduce the third).
- `df.head()`: shows the first 5 rows to check the result.


## 7. Split features (X) and target (y)

- `X` = the columns used to predict
- `y` = the column we want to predict (here: `survived`)


In [ ]:
X = df.drop('survived', axis=1)
y = df['survived']

In [ ]:
X

In [ ]:
y

### Code explanation
- `df.drop('survived', axis=1)`: creates `X` by removing the `survived` column (keeping all the other columns, the ones used to predict). `axis=1` means we're removing a column (not a row).
- `df['survived']`: creates `y`, which contains only the column to predict.


## 8. Split into training and test sets

We train the model on part of the data (80%) and test it on the rest (20%).


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training set size:", X_train.shape)
print("Test set size:", X_test.shape)

### Code explanation
`train_test_split(X, y, test_size=0.2, random_state=42)` splits `X` and `y` into two groups:
- `test_size=0.2`: 20% of the data goes into the test group, 80% into the training group.
- `random_state=42`: fixes the randomness so the split is always the same on every run (reproducible results).

The resulting sizes are then displayed with `.shape`.


## 9. Normalize the numerical data

We put all numerical values on the same scale (useful for many models such as logistic regression, KNN, SVM...).


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
X_train

In [ ]:
X_train_scaled

### Code explanation
- `StandardScaler()` creates a normalization tool.
- `.fit_transform(X_train)`: computes the mean and standard deviation of each column in `X_train`, then transforms the data so it has a mean of 0 and a standard deviation of 1.
- `.transform(X_test)`: applies the **same** transformation (computed on `X_train`) to the test data, without recalculating it — important so the model doesn't "see" the test data in advance.


## ✅ Data ready!

You now have:
- `X_train_scaled`, `X_test_scaled`: the features, normalized
- `y_train`, `y_test`: the values to predict

You can use them directly to train a model, for example:

```python
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train_scaled, y_train)
print("Score:", model.score(X_test_scaled, y_test))
```


## Model explanation

- **LogisticRegression**: a classification algorithm that computes a probability (between 0 and 1) that a passenger survived, then applies a threshold to decide the final category.

- **model.fit(X_train_scaled, y_train)**: trains the model → it learns the relationship between the variables (X_train_scaled) and the actual outcome (y_train).

- **model.score(X_test_scaled, y_test)**: evaluates the model on data it has never seen (X_test, y_test) and returns the percentage of correct predictions (accuracy).


In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train_scaled, y_train)
print("Score :", model.score(X_test_scaled, y_test))

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

We import the necessary tools:
- `accuracy_score`, `precision_score`, `recall_score`, `f1_score`: to compute performance metrics
- `confusion_matrix`: to build the confusion matrix
- `classification_report`: for a full summary of all metrics


In [ ]:
y_pred = model.predict(X_test_scaled)

`model.predict(X_test_scaled)`: the model makes its predictions on the test data (which it never saw during training). The result, `y_pred`, contains a prediction (0 or 1) for each passenger in the test set.


In [ ]:
print("Accuracy  :", round(accuracy_score(y_test, y_pred), 3))
print("Precision :", round(precision_score(y_test, y_pred), 3))
print("Recall    :", round(recall_score(y_test, y_pred), 3))
print("F1-score  :", round(f1_score(y_test, y_pred), 3))

We compare `y_test` (the true answers) to `y_pred` (the model's predictions) to compute 4 metrics:
- **Accuracy**: overall percentage of correct predictions
- **Precision**: among the passengers predicted "survived", how many actually survived
- **Recall**: among the passengers who actually survived, how many did the model find
- **F1-score**: balanced average between precision and recall


In [ ]:
print(classification_report(y_test, y_pred, target_names=["Did not survive", "Survived"]))

`classification_report(...)` automatically displays all the metrics (precision, recall, f1-score) separately for each class ("Survived" / "Did not survive"), plus an overall average.


In [ ]:
cm = confusion_matrix(y_test, y_pred)
print(cm)

`confusion_matrix(y_test, y_pred)` builds a 2x2 table that compares the true values to the predictions:
- Diagonal = correct predictions
- Off-diagonal = errors (false positives / false negatives)


In [ ]:
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=["Did not survive", "Survived"],
            yticklabels=["Did not survive", "Survived"])
plt.xlabel("Prediction")
plt.ylabel("Actual")
plt.title("Confusion matrix")
plt.show()

`sns.heatmap(...)` draws the confusion matrix in color, with the numbers shown in each cell (`annot=True`). The darker a cell, the higher the number. The X axis shows what the model predicted, the Y axis shows the actual truth.


In [ ]:

# Example: a new passenger
new_passenger = pd.DataFrame({
    'pclass': [1],        # 1st class
    'sex': [1],            # 1 = male, 0 = female (encoded as in training)
    'age': [25],
    'sibsp': [0],
    'parch': [0],
    'fare': [80],
    'embarked_Q': [0],
    'embarked_S': [1]      # boarded at Southampton
})

# Important: apply the SAME normalization as on the training data
new_passenger_scaled = scaler.transform(new_passenger)

# Prediction
prediction = model.predict(new_passenger_scaled)
probability = model.predict_proba(new_passenger_scaled)

print("Prediction:", "Survived ✅" if prediction[0] == 1 else "Did not survive ❌")
print("Survival probability:", round(probability[0][1] * 100, 2), "%")

- `prediction[0]`: takes the first (and only) element of the predictions array, since we only tested one passenger.
- `"Survived ✅" if prediction[0] == 1 else "Did not survive ❌"`: translates the number (0 or 1) into readable text.
- `probabilite[0][1]`: retrieves the probability of survival (class `1`) for this passenger, and converts it to a percentage with `* 100`.
